# Bronze to Silver
Reads the raw text out of Bronze. Turns it into real, typed columns.
Removes names and emails. Replaces them with a one-way hash.
Splits true issues away from pull requests (GitHub's issues list has both).
Uses MERGE INTO, so running this twice never makes duplicate rows.

In [0]:
%run ./00_setup_and_logging

# Setup and logging
This notebook does two jobs:
1. Makes the schema (database) and all tables, if they don't exist yet.
2. Gives small helper functions that write one row to the log table every time a step runs.

Run this notebook by itself once. Other notebooks call it with `%run`.

Using workspace.dav_project


## Bronze tables
One table per entity (commits, issues, pulls), plus one quarantine table for
records that could not even be given a key. Every table has `load_timestamp`.

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: pipeline_execution_logs


## Phase 3 upgrade of the log table
The log table already exists from Phase 2, so `CREATE TABLE IF NOT EXISTS`
above does not change it. This cell adds the two new columns to an old
table, only if they are missing. Safe to run any number of times.
- `rows_aggregated`: how many source rows were read and rolled up (Silver to Gold).
- `duration_seconds`: how long the step took.

column already there: rows_aggregated
column already there: duration_seconds


## Small helper functions
`new_run_id()` makes one ID for one notebook run.
`log_run(...)` writes one row to the log table. Call it even when a step fails.
`get_merge_metrics(...)` reads how many rows a MERGE just inserted and updated.

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (blank = process every Bronze row)")
BATCH_ID = dbutils.widgets.get("batch_id").strip() or None
RUN_ID = new_run_id()
FAILED_STEPS = []  # each failing step adds a line here, so the caller can tell
print("batch_id filter:", BATCH_ID or "(none, processing all of Bronze)")
print("run_id:", RUN_ID)

batch_id filter: (none, processing all of Bronze)
run_id: 9428f55b-a569-4409-9294-14d2ae7d2360


## A salt for hashing names and emails
This must stay the SAME every time you run this notebook, or the same
person would get a different hash each run, and MERGE would treat them
as a new person. That's why it's a fixed value in the code, not a widget
you retype. It is not a secret kept safe from the world (this repo is
public); it only stops a casual reader from reversing a hash by guessing
common names and emails directly.

In [0]:
PII_SALT = "dav-project-2026-fixed-salt-change-if-you-want-but-keep-it-the-same"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    BooleanType, ArrayType,
)

## Explicit schemas for the JSON text sitting inside Bronze
These describe only the fields we actually use. A brand-new field GitHub
adds tomorrow is simply ignored here, not an error. A field this schema
expects but doesn't find becomes null, also not an error. See the
schema-drift section near the bottom for how we evolve this on purpose.

In [0]:
COMMIT_JSON_SCHEMA = StructType([
    StructField("sha", StringType()),
    StructField("commit", StructType([
        StructField("message", StringType()),
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
        StructField("committer", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
    ])),
    StructField("author", StructType([StructField("login", StringType())])),
    StructField("committer", StructType([StructField("login", StringType())])),
    StructField("parents", ArrayType(StructType([StructField("sha", StringType())]))),
])

ISSUE_JSON_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("number", IntegerType()),
    StructField("state", StringType()),
    StructField("user", StructType([StructField("login", StringType())])),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("comments", IntegerType()),
    StructField("pull_request", StructType([StructField("url", StringType())])),
])

PULL_JSON_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("number", IntegerType()),
    StructField("state", StringType()),
    StructField("user", StructType([StructField("login", StringType())])),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("merged_at", StringType()),
    StructField("draft", BooleanType()),
    StructField("base", StructType([StructField("ref", StringType())])),
])

TS_FMT = "yyyy-MM-dd'T'HH:mm:ss'Z'"


def hash_key(*cols):
    """sha2 of the salt plus the first non-null value given. Pseudonymous, stable."""
    picked = F.coalesce(*cols, F.lit("unknown"))
    return F.sha2(F.concat(F.lit(PII_SALT), picked), 256)

## Commits: Bronze -> Silver

In [0]:
def silver_commits():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_commits")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), COMMIT_JSON_SCHEMA))

        bad = parsed.filter(F.col("data").isNull() | F.col("data.sha").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.sha").isNotNull())

        silver = good.select(
            F.col("data.sha").alias("sha"),
            F.col("source_repo").alias("repo"),
            hash_key(F.col("data.author.login"), F.col("data.commit.author.email")).alias("author_key"),
            hash_key(F.col("data.committer.login"), F.col("data.commit.committer.email")).alias("committer_key"),
            F.to_timestamp(F.col("data.commit.author.date"), TS_FMT).alias("committed_at"),
            F.length(F.col("data.commit.message")).alias("message_length"),
            (F.size(F.coalesce(F.col("data.parents"), F.array())) > 1).alias("is_merge_commit"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["sha"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_commits_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_commit"
            # The 8 columns are written by name on purpose (not UPDATE SET *).
            # silver_commit may have an extra column that this batch does not have
            # (has_extra_stats, added by 03_schema_drift_demo). With "SET *" that
            # extra column makes the MERGE fail. Naming the columns avoids it, and
            # the extra column is simply left as it is (empty for new rows).
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_commits_batch AS s
                ON t.sha = s.sha
                WHEN MATCHED THEN UPDATE SET
                    t.repo = s.repo, t.author_key = s.author_key, t.committer_key = s.committer_key,
                    t.committed_at = s.committed_at, t.message_length = s.message_length,
                    t.is_merge_commit = s.is_merge_commit, t.load_timestamp = s.load_timestamp
                WHEN NOT MATCHED THEN INSERT
                    (sha, repo, author_key, committer_key, committed_at, message_length,
                     is_merge_commit, load_timestamp)
                    VALUES (s.sha, s.repo, s.author_key, s.committer_key, s.committed_at,
                            s.message_length, s.is_merge_commit, s.load_timestamp)
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("commit").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse commit JSON or missing sha").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "commits", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "commits", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: commits:", e)
        FAILED_STEPS.append(f"commits: {e}")
        return 0, 0, 0

## Issues: Bronze -> Silver
GitHub's issues list includes pull requests too. A real issue has no
`pull_request` field. We drop anything that has one, since the pulls
table already has that record, with more detail.

In [0]:
def silver_issues():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_issues")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), ISSUE_JSON_SCHEMA))

        is_real_issue = F.col("data.pull_request").isNull()
        bad = parsed.filter(F.col("data").isNull() | F.col("data.id").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.id").isNotNull() & is_real_issue)

        closed_at_ts = F.to_timestamp(F.col("data.closed_at"), TS_FMT)
        created_at_ts = F.to_timestamp(F.col("data.created_at"), TS_FMT)

        silver = good.select(
            F.col("data.id").cast("string").alias("issue_id"),
            F.col("source_repo").alias("repo"),
            F.col("data.number").alias("number"),
            F.col("data.state").alias("state"),
            hash_key(F.col("data.user.login")).alias("author_key"),
            created_at_ts.alias("created_at"),
            F.to_timestamp(F.col("data.updated_at"), TS_FMT).alias("updated_at"),
            closed_at_ts.alias("closed_at"),
            F.col("data.comments").alias("comment_count"),
            F.when(closed_at_ts.isNotNull(),
                   (F.unix_timestamp(closed_at_ts) - F.unix_timestamp(created_at_ts)) / 3600.0
                   ).alias("time_to_close_hours"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["issue_id"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_issues_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_issue"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_issues_batch AS s
                ON t.issue_id = s.issue_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("issue").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse issue JSON or missing id").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "issues", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "issues", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: issues:", e)
        FAILED_STEPS.append(f"issues: {e}")
        return 0, 0, 0

## Pull requests: Bronze -> Silver

In [0]:
def silver_pulls():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_pulls")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), PULL_JSON_SCHEMA))

        bad = parsed.filter(F.col("data").isNull() | F.col("data.id").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.id").isNotNull())

        created_at_ts = F.to_timestamp(F.col("data.created_at"), TS_FMT)
        merged_at_ts = F.to_timestamp(F.col("data.merged_at"), TS_FMT)

        silver = good.select(
            F.col("data.id").cast("string").alias("pr_id"),
            F.col("source_repo").alias("repo"),
            F.col("data.number").alias("number"),
            F.col("data.state").alias("state"),
            hash_key(F.col("data.user.login")).alias("author_key"),
            created_at_ts.alias("created_at"),
            F.to_timestamp(F.col("data.updated_at"), TS_FMT).alias("updated_at"),
            F.to_timestamp(F.col("data.closed_at"), TS_FMT).alias("closed_at"),
            merged_at_ts.alias("merged_at"),
            merged_at_ts.isNotNull().alias("is_merged"),
            F.coalesce(F.col("data.draft"), F.lit(False)).alias("is_draft"),
            F.col("data.base.ref").alias("base_branch"),
            F.when(merged_at_ts.isNotNull(),
                   (F.unix_timestamp(merged_at_ts) - F.unix_timestamp(created_at_ts)) / 3600.0
                   ).alias("time_to_merge_hours"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["pr_id"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_pulls_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_pull_request"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_pulls_batch AS s
                ON t.pr_id = s.pr_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("pull_request").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse pull request JSON or missing id").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "pulls", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "pulls", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: pulls:", e)
        FAILED_STEPS.append(f"pulls: {e}")
        return 0, 0, 0

## Run all three

In [0]:
print("commits:", silver_commits())
print("issues:", silver_issues())
print("pulls:", silver_pulls())

[log] Bronze-to-Silver / commits -> SUCCESS (inserted=3, updated=13840, quarantined=1, aggregated=0, seconds=13.5)
commits: (3, 13840, 1)
[log] Bronze-to-Silver / issues -> SUCCESS (inserted=3, updated=8835, quarantined=0, aggregated=0, seconds=9.6)
issues: (3, 8835, 0)
[log] Bronze-to-Silver / pulls -> SUCCESS (inserted=2, updated=18156, quarantined=0, aggregated=0, seconds=10.2)
pulls: (2, 18156, 0)


In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,start_time
9428f55b-a569-4409-9294-14d2ae7d2360,Bronze-to-Silver,commits,SUCCESS,3,13840,1,2026-10-24T14:34:05.459Z
9428f55b-a569-4409-9294-14d2ae7d2360,Bronze-to-Silver,issues,SUCCESS,3,8835,0,2026-10-24T14:34:20.777Z
9428f55b-a569-4409-9294-14d2ae7d2360,Bronze-to-Silver,pulls,SUCCESS,2,18156,0,2026-10-24T14:34:31.963Z


In [0]:
# Tell whoever started this notebook (05_run_pipeline) how it went.
import json
dbutils.notebook.exit(json.dumps({"all_ok": not FAILED_STEPS, "problems": FAILED_STEPS}))